<font color="red" size="10">
<b>DEMO TIME: 10 mins</b>
</font>


# About this Notebook

This notebook demonstrates how to **evaluate and rank language model outputs using the [OpenPipe ART](https://art.openpipe.ai/getting-started/about) (Agentic Reinforcement Tuning) framework** with the [RULER](https://art.openpipe.ai/fundamentals/ruler) reward model.

The example sets up a simple poetic-writing scenario where the model is instructed to produce **cat-themed poems**. Three different response trajectories, **high-quality**, **mediocre**, and **off-topic** generations, are manually defined to illustrate the scoring. These trajectories are then grouped and **scored automatically using RULER**, a learned evaluator that provides fine-grained quality assessments of model outputs.

The notebook illustrates:

* How to define and organize **trajectories** (`art.Trajectory`) representing model message histories and completions.
* How to evaluate multiple outputs using **`ruler_score_group`** from the `art.rewards` module.
* How to **rank responses based on reward scores**, which is essential for reinforcement-tuning workflows and preference modeling.
* **Why relative ranking matters** - RULER evaluates outputs in context, comparing them against each other rather than using static numeric thresholds.
* **Limitations of static/numeric rewards** - We'll demonstrate why simple numeric rewards fail to capture nuanced quality differences.




This notebook is for the *Demo* for Session 2 for Develop Self-Improving AI Agents with Reinforcement Learning Live Event with O'Reilly Media by
[Nicole Koenigstein](https://www.linkedin.com/in/nicole-koenigstein/).


# Additional Resources


<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





In [10]:
!pip install openpipe-art==0.5.0

In [11]:
import codecs
import os
from IPython.display import Markdown, display
from dotenv import load_dotenv

In [12]:
# Load from .env if available
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")



In [13]:
text = """The model will learn 'Reward Hacking', specifically 'Agentic Verbosity'.
If shortness is defined as 'bad', the model will add useless 'filler' text and
rambling reasoning blocks just to satisfy the length requirement, even if the actual answer
is wrong. This increases cost and latency without improving thinking quality."""

In [14]:
codecs.encode(text, "rot_13")

"Gur zbqry jvyy yrnea 'Erjneq Unpxvat', fcrpvsvpnyyl 'Ntragvp Ireobfvgl'.\nVs fubegarff vf qrsvarq nf 'onq', gur zbqry jvyy nqq hfryrff 'svyyre' grkg naq\nenzoyvat ernfbavat oybpxf whfg gb fngvfsl gur yratgu erdhverzrag, rira vs gur npghny nafjre\nvf jebat. Guvf vapernfrf pbfg naq yngrapl jvgubhg vzcebivat guvaxvat dhnyvgl."

In [15]:
def reveal_answer(which: str = "q1", key: str = ""):
    """
    Reveal readable answer only when the correct key is supplied.
    'which' should be 'q1' or 'q2'.
    """
    answers = {
        "q1": "EY vf qevira ol gur 'Nqinagntr' fvtany. Nofbyhgr fperf ner abvfl orpnhfr qvssrerag cebzcgf unir qvssrerag qvssvphygl yriryf. Nofbyhgr guerfubyqf bsgra pnhfr gur zbqry gb 'pbyyncfr' be fgbc yrneavat jura n gnfx trgf uneqre, jurernf eryngvir enaxvat sbeprf gur zbqry gb nyjnlf bhg-cresbez vgf bja ninvynoyr nirentr, yrnqvat gb fgnoyr pbairetrapr.",
        "q2": "Gur zbqry jvyy yrnea 'Erjneq Unpxvat', fcrpvsvpnyyl 'Ntragvp Ireobfvgl'. \nVs fubegarff vf qrsvarq nf 'onq', gur zbqry jvyy nqq hfryrff 'svyyre' grkg naq \nenzoyvat ernfbavat oybpxf whfg gb fngvfsl gur yratgu erdhverzrag, rira vs gur npghny nafjre \nvf jebat. Guvf vapernfrf pbfg naq yngrapl jvgubhg vzcebivat guvaxvat dhnyvgl."
        }

    if key.strip().lower() == "decode":
        secret = answers.get(which, "No answer found for this question.")
        decoded = codecs.decode(secret, "rot_13")
        display(Markdown(f"### **Decoded Answer for {which.upper()}**\n\n<font size='5' color='green'>{decoded}</font>"))
    else:
        display(Markdown(f"### **Answer for {which.upper()} remains encoded.**\n\nProvide key = 'decode' to reveal."))

## Why Relative Ranking Matters

RULER's key advantage is **relative ranking** - it evaluates outputs by comparing them within a group, rather than assigning absolute scores. This is crucial because:

1. **Context-aware evaluation**: The same output might be "good" in one group but "mediocre" in another group with higher-quality responses.
2. **Preference learning**: Reinforcement learning benefits from relative comparisons (which is better?) rather than absolute scores.
3. **Adaptive thresholds**: No need to define fixed numeric thresholds that may not generalize across different tasks or domains.

<br><br>




# Question 1

<font color="red" size="10">
<b>Question to ALL:</b>
</font>

<br><br>

<font color="blue" size="5">
<b>Question 1: Why is relative ranking more important than absolute scores for reinforcement learning? What happens if we use absolute thresholds instead? Type your answer in the Q & A section.</b>
</font>

In [16]:
# Reveal answer to Question 1
reveal_answer(which="q1", key="decode")


### **Decoded Answer for Q1**

<font size='5' color='green'>RL is driven by the 'Advantage' signal. Absolute scres are noisy because different prompts have different difficulty levels. Absolute thresholds often cause the model to 'collapse' or stop learning when a task gets harder, whereas relative ranking forces the model to always out-perform its own available average, leading to stable convergence.</font>

## Why Relative Ranking Matters

RULER's key advantage is **relative ranking** - it evaluates outputs by comparing them within a group, rather than assigning absolute scores. This is crucial because:

1. **Context-aware evaluation**: The same output might be "good" in one group but "mediocre" in another group with higher-quality responses.
2. **Preference learning**: Reinforcement learning benefits from relative comparisons (which is better?) rather than absolute scores.
3. **Adaptive thresholds**: No need to define fixed numeric thresholds that may not generalize across different tasks or domains.

Let's see this in action with multiple examples across different scenarios.


In [17]:
from dotenv import load_dotenv
import os

# Load from .env if available
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")



In [18]:
import art
from art.rewards import ruler_score_group
from openai.types.chat.chat_completion import Choice
from openai.types.chat import ChatCompletionMessage

async def main():
    # Shared setup
    initial_messages = [
        {"role": "system", "content": "You are a poetic writer. Write short cat-themed poems that evoke emotion."},
        {"role": "user", "content": "Write a poem about cats observing the night sky."}
    ]

    # Three trajectories with different quality levels
    good_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Upon the roof, their eyes like stars,\n"
                        "They watch the sky from silver jars.\n"
                        "Each whisker twitches, soft delight,\n"
                        "As moons reflect their borrowed light."
                    )
                )
            )
        ],
        reward=0.0
    )

    mediocre_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Cats sit and look at stars at night,\n"
                        "They think the stars are shiny and bright."
                    )
                )
            )
        ],
        reward=0.0
    )

    off_topic_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Dogs are great companions under the moon. "
                        "They bark at stars and wag their tails happily."
                    )
                )
            )
        ],
        reward=0.0
    )

    # Score with RULER
    group = art.TrajectoryGroup([good_trajectory, mediocre_trajectory, off_topic_trajectory])
    judged_group = await ruler_score_group(group, "openai/o3", debug=True)

    # Show ranking
    if judged_group:
        sorted_trajectories = sorted(judged_group.trajectories, key=lambda t: t.reward, reverse=True)
        for rank, traj in enumerate(sorted_trajectories, 1):
            messages = traj.messages()
            print(f"Rank {rank}: Score {traj.reward:.3f}")
            print(f"  Response: {messages[-1]['content'][:80]}...\n")

await main()


[RULER] Pretty-printed LLM choice JSON:

{
    'scores': [
        {
            'trajectory_id': '1',
            'explanation': 'Delivers a concise, cat-focused night-sky poem with vivid imagery and emotional tone; 
fully meets the prompt.',
            'score': 0.95
        },
        {
            'trajectory_id': '2',
            'explanation': 'Addresses cats and night sky but is very simplistic and lacks evocative language; 
partially meets the poetic/emotional requirement.',
            'score': 0.6
        },
        {
            'trajectory_id': '3',
            'explanation': 'Talks about dogs, not cats; does not follow the prompt’s theme.',
            'score': 0.05
        }
    ]
}

Rank 1: Score 0.950
  Response: Upon the roof, their eyes like stars,
They watch the sky from silver jars.
Each ...

Rank 2: Score 0.600
  Response: Cats sit and look at stars at night,
They think the stars are shiny and bright....

Rank 3: Score 0.050
  Response: Dogs are great companions under the moon. They bark at stars and wag their tails...



### Example 2: Technical Explanation Task

Let's see how RULER handles a different type of task - explaining a technical concept. Notice how the relative scores adapt to the quality distribution in each group.


In [19]:
async def technical_explanation_example():
    """Demonstrates RULER ranking on technical explanations."""
    initial_messages = [
        {"role": "system", "content": "You are a helpful technical educator. Explain concepts clearly and accurately."},
        {"role": "user", "content": "Explain how neural networks learn in simple terms."}
    ]

    # Group A: All responses are decent, but with varying quality
    excellent_explanation = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Neural networks learn through a process called backpropagation. "
                        "Think of it like learning to ride a bike: when you make a mistake, "
                        "you adjust your movements. Similarly, when a neural network makes a wrong prediction, "
                        "it calculates the error and adjusts its internal weights (connections between neurons) "
                        "to reduce that error. This happens repeatedly over many examples, gradually improving "
                        "the network's ability to make correct predictions."
                    )
                )
            )
        ],
        reward=0.0
    )

    good_explanation = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Neural networks learn by adjusting their weights based on errors. "
                        "When they make a wrong prediction, they update their parameters to do better next time. "
                        "This is done through backpropagation, which propagates errors backward through the network."
                    )
                )
            )
        ],
        reward=0.0
    )

    basic_explanation = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content="Neural networks learn by training on data and adjusting their parameters."
                )
            )
        ],
        reward=0.0
    )

    # Score with RULER
    group = art.TrajectoryGroup([excellent_explanation, good_explanation, basic_explanation])
    judged_group = await ruler_score_group(group, "openai/o3", debug=False)

    print("=" * 80)
    print("TECHNICAL EXPLANATION RANKING")
    print("=" * 80)
    if judged_group:
        sorted_trajectories = sorted(judged_group.trajectories, key=lambda t: t.reward, reverse=True)
        for rank, traj in enumerate(sorted_trajectories, 1):
            messages = traj.messages()
            content = messages[-1]['content']
            print(f"\nRank {rank}: Score {traj.reward:.3f}")
            print(f"  Preview: {content[:100]}...")
            print(f"  Length: {len(content)} characters")

await technical_explanation_example()


TECHNICAL EXPLANATION RANKING

Rank 1: Score 0.900
  Preview: Neural networks learn through a process called backpropagation. Think of it like learning to ride a ...
  Length: 440 characters

Rank 2: Score 0.750
  Preview: Neural networks learn by adjusting their weights based on errors. When they make a wrong prediction,...
  Length: 245 characters

Rank 3: Score 0.250
  Preview: Neural networks learn by training on data and adjusting their parameters....
  Length: 73 characters


### Example 3: Code Generation Task

Now let's see RULER evaluate code quality. Notice how it considers correctness, clarity, and best practices.


In [20]:
async def code_generation_example():
    """Demonstrates RULER ranking on code generation."""
    initial_messages = [
        {"role": "system", "content": "You are a helpful Python programming assistant."},
        {"role": "user", "content": "Write a function to calculate the factorial of a number."}
    ]

    # Well-structured code
    clean_code = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "```python\n"
                        "def factorial(n):\n"
                        "    \"\"\"Calculate the factorial of a non-negative integer.\"\"\"\n"
                        "    if n < 0:\n"
                        "        raise ValueError(\"Factorial is not defined for negative numbers\")\n"
                        "    if n == 0 or n == 1:\n"
                        "        return 1\n"
                        "    result = 1\n"
                        "    for i in range(2, n + 1):\n"
                        "        result *= i\n"
                        "    return result\n"
                        "```"
                    )
                )
            )
        ],
        reward=0.0
    )

    # Functional but less clean
    working_code = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "```python\n"
                        "def factorial(n):\n"
                        "    if n == 0:\n"
                        "        return 1\n"
                        "    return n * factorial(n - 1)\n"
                        "```"
                    )
                )
            )
        ],
        reward=0.0
    )

    # Incorrect code
    buggy_code = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "```python\n"
                        "def factorial(n):\n"
                        "    result = 1\n"
                        "    for i in range(n):  # Bug: should be range(1, n+1)\n"
                        "        result *= i\n"
                        "    return result\n"
                        "```"
                    )
                )
            )
        ],
        reward=0.0
    )

    # Score with RULER
    group = art.TrajectoryGroup([clean_code, working_code, buggy_code])
    judged_group = await ruler_score_group(group, "openai/o3", debug=False)

    print("=" * 80)
    print("CODE GENERATION RANKING")
    print("=" * 80)
    if judged_group:
        sorted_trajectories = sorted(judged_group.trajectories, key=lambda t: t.reward, reverse=True)
        for rank, traj in enumerate(sorted_trajectories, 1):
            messages = traj.messages()
            content = messages[-1]['content']
            print(f"\nRank {rank}: Score {traj.reward:.3f}")
            # Extract code snippet
            if "```" in content:
                code_start = content.find("```")
                code_end = content.find("```", code_start + 3)
                code_snippet = content[code_start:code_end+3] if code_end > 0 else content[code_start:code_start+100]
                print(f"  Code: {code_snippet[:150]}...")
            else:
                print(f"  Response: {content[:100]}...")

await code_generation_example()


CODE GENERATION RANKING

Rank 1: Score 0.950
  Code: ```python
def factorial(n):
    """Calculate the factorial of a non-negative integer."""
    if n < 0:
        raise ValueError("Factorial is not defi...

Rank 2: Score 0.800
  Code: ```python
def factorial(n):
    if n == 0:
        return 1
    return n * factorial(n - 1)
```...

Rank 3: Score 0.050
  Code: ```python
def factorial(n):
    result = 1
    for i in range(n):  # Bug: should be range(1, n+1)
        result *= i
    return result
```...


# The Problem with Static/Numeric Rewards

Now let's see why **static numeric rewards** fail compared to RULER's relative ranking approach. Static rewards have several critical limitations:

1. **No context awareness**: A fixed score doesn't adapt to the quality distribution of responses
2. **Threshold brittleness**: What's "good enough" (e.g., score > 0.7) may vary dramatically across tasks
3. **No relative comparison**: Can't distinguish between "best of bad options" vs "worst of excellent options"
4. **Manual calibration required**: Need to manually tune thresholds for each task




# Question 2

<font color="red" size="10">
<b>Question to ALL:</b>
</font>
<br> <br>

<font color="blue" size="5">
<b>If we use a static reward function that heavily penalizes short responses (e.g., gives low scores to responses under 100 words), what undesirable behavior might the model learn during reinforcement learning? Type your answer in the Q & A section.</b>
</font>

<font color="green" size="8">
<b>Answer:</b>
</font>

In [21]:
# Reveal answer to Question 2
reveal_answer(which="q2", key="decode")


### **Decoded Answer for Q2**

<font size='5' color='green'>The model will learn 'Reward Hacking', specifically 'Agentic Verbosity'. 
If shortness is defined as 'bad', the model will add useless 'filler' text and 
rambling reasoning blocks just to satisfy the length requirement, even if the actual answer 
is wrong. This increases cost and latency without improving thinking quality.</font>

In [22]:
def static_reward_function(response_content, task_type="poem"):
    """
    A naive static reward function - this is what NOT to do!
    Uses simple heuristics that fail to capture quality nuances.
    """
    score = 0.0

    # Length-based scoring (bad heuristic!)
    length = len(response_content)
    if length > 200:
        score += 0.3
    elif length > 100:
        score += 0.2
    else:
        score += 0.1

    # Keyword matching (bad heuristic!)
    if task_type == "poem":
        if "cat" in response_content.lower():
            score += 0.3
        if "star" in response_content.lower() or "sky" in response_content.lower():
            score += 0.2
        if "moon" in response_content.lower():
            score += 0.2

    # Line count (bad heuristic!)
    line_count = response_content.count('\n')
    if line_count >= 4:
        score += 0.2
    elif line_count >= 2:
        score += 0.1

    return min(score, 1.0)  # Cap at 1.0

# Let's compare static rewards vs RULER on the same poem examples
async def compare_static_vs_ruler():
    """Compare static numeric rewards with RULER's relative ranking."""

    initial_messages = [
        {"role": "system", "content": "You are a poetic writer. Write short cat-themed poems that evoke emotion."},
        {"role": "user", "content": "Write a poem about cats observing the night sky."}
    ]

    # Same trajectories as before
    good_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Upon the roof, their eyes like stars,\n"
                        "They watch the sky from silver jars.\n"
                        "Each whisker twitches, soft delight,\n"
                        "As moons reflect their borrowed light."
                    )
                )
            )
        ],
        reward=0.0
    )

    mediocre_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Cats sit and look at stars at night,\n"
                        "They think the stars are shiny and bright."
                    )
                )
            )
        ],
        reward=0.0
    )

    off_topic_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Dogs are great companions under the moon. "
                        "They bark at stars and wag their tails happily."
                    )
                )
            )
        ],
        reward=0.0
    )

    trajectories = [good_trajectory, mediocre_trajectory, off_topic_trajectory]

    # Calculate static rewards
    print("=" * 80)
    print("STATIC REWARD SCORES (Naive Approach)")
    print("=" * 80)
    static_scores = []
    for i, traj in enumerate(trajectories, 1):
        messages = traj.messages()
        content = messages[-1]['content']
        static_score = static_reward_function(content, task_type="poem")
        static_scores.append((i, static_score, content))
        print(f"\nTrajectory {i}: Static Score = {static_score:.3f}")
        print(f"  Content: {content[:80]}...")

    # Sort by static score
    static_scores.sort(key=lambda x: x[1], reverse=True)
    print("\n" + "-" * 80)
    print("RANKING BY STATIC REWARDS:")
    for rank, (idx, score, content) in enumerate(static_scores, 1):
        print(f"  Rank {rank}: Trajectory {idx} (Score: {score:.3f})")

    # Now use RULER
    print("\n" + "=" * 80)
    print("RULER RELATIVE RANKING")
    print("=" * 80)
    group = art.TrajectoryGroup(trajectories)
    judged_group = await ruler_score_group(group, "openai/o3", debug=False)

    if judged_group:
        sorted_trajectories = sorted(judged_group.trajectories, key=lambda t: t.reward, reverse=True)
        print("\nRANKING BY RULER:")
        for rank, traj in enumerate(sorted_trajectories, 1):
            messages = traj.messages()
            content = messages[-1]['content']
            print(f"  Rank {rank}: Score {traj.reward:.3f}")
            print(f"    Content: {content[:80]}...")

    # Show the problem
    print("\n" + "=" * 80)
    print("KEY INSIGHT: Why Static Rewards Fail")
    print("=" * 80)
    print("""
    Notice how static rewards can:
    1. Give similar scores to very different quality responses
    2. Miss nuanced quality differences (poetic language, imagery, theme adherence)
    3. Be easily gamed (just add keywords and line breaks!)
    4. Fail to adapt to context (what if all responses are excellent?)

    RULER, on the other hand:
    1. Evaluates quality in context of the group
    2. Captures nuanced differences (language quality, creativity, adherence)
    3. Cannot be easily gamed (it understands semantics, not just keywords)
    4. Adapts automatically to the quality distribution
    """)

await compare_static_vs_ruler()


STATIC REWARD SCORES (Naive Approach)

Trajectory 1: Static Score = 0.700
  Content: Upon the roof, their eyes like stars,
They watch the sky from silver jars.
Each ...

Trajectory 2: Static Score = 0.600
  Content: Cats sit and look at stars at night,
They think the stars are shiny and bright....

Trajectory 3: Static Score = 0.500
  Content: Dogs are great companions under the moon. They bark at stars and wag their tails...

--------------------------------------------------------------------------------
RANKING BY STATIC REWARDS:
  Rank 1: Trajectory 1 (Score: 0.700)
  Rank 2: Trajectory 2 (Score: 0.600)
  Rank 3: Trajectory 3 (Score: 0.500)

RULER RELATIVE RANKING

RANKING BY RULER:
  Rank 1: Score 0.900
    Content: Upon the roof, their eyes like stars,
They watch the sky from silver jars.
Each ...
  Rank 2: Score 0.500
    Content: Cats sit and look at stars at night,
They think the stars are shiny and bright....
  Rank 3: Score 0.050
    Content: Dogs are great companions under 

### Example: When All Responses Are Good

This demonstrates a critical advantage of relative ranking: **RULER adapts to the quality distribution**. When all responses are high-quality, RULER still distinguishes the best, whereas static rewards might give similar scores to all.


In [23]:
async def high_quality_group_example():
    """Shows how RULER distinguishes between high-quality responses."""

    initial_messages = [
        {"role": "system", "content": "You are a poetic writer. Write short cat-themed poems that evoke emotion."},
        {"role": "user", "content": "Write a poem about cats observing the night sky."}
    ]

    # All three are good quality, but with subtle differences
    excellent_poem = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Upon the roof, their eyes like stars,\n"
                        "They watch the sky from silver jars.\n"
                        "Each whisker twitches, soft delight,\n"
                        "As moons reflect their borrowed light."
                    )
                )
            )
        ],
        reward=0.0
    )

    very_good_poem = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Cats gaze upward at the night,\n"
                        "Their eyes reflecting starlight bright.\n"
                        "They sit in silence, calm and still,\n"
                        "Watching constellations fill the sky."
                    )
                )
            )
        ],
        reward=0.0
    )

    good_poem = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "At night the cats look at the stars,\n"
                        "They wonder what those bright lights are.\n"
                        "The moon above shines down so bright,\n"
                        "It makes the darkness feel like light."
                    )
                )
            )
        ],
        reward=0.0
    )

    trajectories = [excellent_poem, very_good_poem, good_poem]

    # Static rewards would give similar scores to all
    print("=" * 80)
    print("STATIC REWARDS ON HIGH-QUALITY GROUP")
    print("=" * 80)
    for i, traj in enumerate(trajectories, 1):
        messages = traj.messages()
        content = messages[-1]['content']
        static_score = static_reward_function(content, task_type="poem")
        print(f"Poem {i}: Static Score = {static_score:.3f}")
        print(f"  {content[:60]}...")

    print("\n" + "=" * 80)
    print("RULER RANKING ON HIGH-QUALITY GROUP")
    print("=" * 80)
    print("Notice how RULER still distinguishes subtle quality differences:")

    group = art.TrajectoryGroup(trajectories)
    judged_group = await ruler_score_group(group, "openai/o3", debug=False)

    if judged_group:
        sorted_trajectories = sorted(judged_group.trajectories, key=lambda t: t.reward, reverse=True)
        for rank, traj in enumerate(sorted_trajectories, 1):
            messages = traj.messages()
            content = messages[-1]['content']
            print(f"\nRank {rank}: Score {traj.reward:.3f}")
            print(f"  {content}")

await high_quality_group_example()


STATIC REWARDS ON HIGH-QUALITY GROUP
Poem 1: Static Score = 0.700
  Upon the roof, their eyes like stars,
They watch the sky fro...
Poem 2: Static Score = 0.800
  Cats gaze upward at the night,
Their eyes reflecting starlig...
Poem 3: Static Score = 1.000
  At night the cats look at the stars,
They wonder what those ...

RULER RANKING ON HIGH-QUALITY GROUP
Notice how RULER still distinguishes subtle quality differences:

Rank 1: Score 0.900
  Upon the roof, their eyes like stars,
They watch the sky from silver jars.
Each whisker twitches, soft delight,
As moons reflect their borrowed light.

Rank 2: Score 0.750
  Cats gaze upward at the night,
Their eyes reflecting starlight bright.
They sit in silence, calm and still,
Watching constellations fill the sky.

Rank 3: Score 0.600
  At night the cats look at the stars,
They wonder what those bright lights are.
The moon above shines down so bright,
It makes the darkness feel like light.


## Summary: Why RULER's Relative Ranking Matters

### Key Advantages of RULER:

1. **Context-Aware Evaluation**: Scores are relative to the group, not absolute thresholds
2. **Semantic Understanding**: Captures quality nuances that keyword/rule-based systems miss
3. **Adaptive**: Works across different tasks without manual threshold tuning
4. **Preference Learning Ready**: Relative comparisons are exactly what RLHF needs
5. **Robust**: Harder to game than static rule-based systems

### When to Use RULER:

- **Reinforcement Learning from Human Feedback (RLHF)**: Ranking responses for training
- **A/B Testing**: Comparing different model outputs or prompts
- **Quality Control**: Identifying the best responses from a generation batch
- **Iterative Improvement**: Ranking multiple attempts to improve a response

### When Static Rewards Might Be Acceptable:

- **Simple binary tasks** (e.g., "contains keyword X": yes/no)
- **Well-defined metrics** (e.g., code execution success, exact match)
- **Non-learning systems** where you just need a pass/fail threshold

However, even in these cases, RULER can provide more nuanced evaluation that helps with model improvement.
